# Lesson 11 - मॉडेल कॉन्टेक्स्ट प्रोटोकॉल (MCP)

The **Model Context Protocol (MCP)** is an open standard that enables agents to dynamically discover and use tools, resources, and data sources at runtime. Instead of hardcoding tools into an agent, MCP lets agents connect to external servers that expose capabilities on demand.

In this lesson, you'll learn:
- MCP काय आहे आणि एजंट प्रणालींसाठी ते का महत्त्वाचे आहे
- MCP क्लायंट-सर्व्हर आर्किटेक्चर कसे कार्य करते
- MCP-शैलीच्या साधन शोधाचा वापर करणारे एजंट कसे तयार करायचे


## सेटअप

**पूर्व-आवश्यकता:**
- तैनात केलेला मॉडेल असलेला Azure AI Foundry प्रकल्प
- प्रमाणीकरणासाठी `az login` चालवा


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Model Context Protocol (MCP) म्हणजे काय?

MCP एआय एजंट्सना बाह्य साधने आणि डेटा स्रोत शोधण्यासाठी व त्यांच्याशी संवाद साधण्यासाठी एक मानक पद्धत परिभाषित करते:

- **MCP Server**: मानक प्रोटोकॉलद्वारे साधने, संसाधने आणि प्रॉम्प्ट उपलब्ध करून देते
- **MCP Client**: जो सर्व्हरशी कनेक्ट होतो आणि उपलब्ध क्षमता ओळखतो असे एजंट रनटाइम
- **Dynamic Discovery**: एजंट्सना हार्डकोड केलेल्या साधनांची गरज नसते — ते रनटाइममध्ये काय उपलब्ध आहे ते शोधतात

हे विस्तारयोग्य एजंट सिस्टीम तयार करताना प्रभावी आहे, जिथे नवीन क्षमता एजंटच्या कोडामध्ये बदल न करता जोडता येतात.


## MCP कसे कार्य करते

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. एजंट (MCP client) एका MCP server शी कनेक्ट होतो
2. सर्व्हर उपलब्ध साधने आणि त्यांच्या स्कीमांची यादीसह प्रतिसाद देतो
3. मग एजंट त्याच्या तर्कसंगती दरम्यान शोधलेल्या कोणत्याही साधनाला कॉल करू शकतो
4. परिणाम त्याच प्रोटोकॉलमार्गे परत येतात


## MCP टूल शोधाचे अनुकरण

खऱ्या MCP सर्व्हरसाठी एक चालू सर्व्हर प्रक्रिया आवश्यक असल्याने, आम्ही `@tool` फंक्शन्स वापरून हा नमुना दाखवू, जे MCP-शी जोडलेली निवास सेवा काय पुरवेल ते अनुकरण करतात.

उत्पादनात, हे टूल्स स्थानिकरित्या परिभाषित करण्याऐवजी MCP सर्व्हरकडून गतिशीलरित्या शोधले जातील.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## MCP-शैलीतील साधने वापरून एजंट तयार करणे


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## उत्पादनातील MCP

उत्पादनाच्या वातावरणात, MCP काही शक्तिशाली पॅटर्न सक्षम करते:

- **डायनॅमिक टूल शोध**: एजंट MCP सर्व्हरशी कनेक्ट होतात आणि रनटाइममध्ये टूल्स शोधतात
- **विभक्त आर्किटेक्चर**: टूल प्रदाते एजंटपासून स्वतंत्रपणे अद्यतनित करू शकतात
- **संस्थांदरम्यान सामायिकरण**: टीम्स MCP सर्व्हर्सद्वारे क्षमता उघड करू शकतात ज्याचा कोणताही एजंट वापर करू शकतो
- **Microsoft Agent Framework समर्थन**: MAF मध्ये `mcp` इंटिग्रेशनद्वारे अंगभूत MCP क्लायंट समर्थन आहे

MAF सह प्रत्यक्ष MCP सर्व्हर वापरण्यासाठी, तुम्ही `hosted_mcp_tool()` किंवा MCP क्लायंट इंटिग्रेशनद्वारे कनेक्ट कराल.

**अधिक जाणून घ्या:**
- [MCP स्पेसिफिकेशन](https://modelcontextprotocol.io/)
- [Microsoft Agent Framework MCP समर्थन](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## सारांश

या धड्यात तुम्ही शिकलात:
- **MCP** हे एजंट्स आणि टूल पुरवठादारांदरम्यान डायनॅमिक साधन शोधासाठीचे एक खुले मानक आहे
- हे **क्लायंट-सर्व्हर आर्किटेक्चर** एजंटांना रनटाइमवर क्षमता शोधण्याची परवानगी देते
- MCP सक्षम करते **विस्तारयोग्य, विभक्त एजंट प्रणाली** जिथे साधने कोड बदलावाशिवाय जोडली जाऊ शकतात
- Microsoft Agent Framework प्रदान करते **अंतर्निर्मित MCP समर्थन** उत्पादन वापरासाठी


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
अस्वीकरण:
हा दस्तऐवज AI अनुवाद सेवा Co‑op Translator (https://github.com/Azure/co-op-translator) वापरून अनुवादित केला गेला आहे. आम्ही अचूकतेसाठी प्रयत्न करतो, परंतु कृपया लक्षात घ्या की स्वयंचलित अनुवादांमध्ये त्रुटी किंवा अयोग्य माहिती असू शकते. मूळ दस्तऐवज त्याच्या मूळ भाषेत अधिकृत स्रोत मानला जावा. महत्त्वाच्या माहितीसाठी व्यावसायिक मानवी अनुवादाची शिफारस केली जाते. या अनुवादाच्या वापरामुळे उद्भवलेल्या कोणत्याही गैरसमजुतींसाठी किंवा चुकीच्या अर्थनिर्णयासाठी आम्ही जबाबदार नाही.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
